**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 15: 합성 데이터 생성 & Distillation

**Part 3: 파인튜닝 기초** | GPU: No (API 기반)

---

### 📋 학습 목표
- 🎯 합성 데이터(Synthetic Data) 생성의 필요성과 방법론을 이해한다
- 🎯 Self-Instruct 방법론의 원리를 파악한다
- 🎯 GPT-4를 활용하여 학습 데이터를 자동 생성한다
- 🎯 Knowledge Distillation의 개념과 활용법을 학습한다
- 🎯 생성 데이터의 품질 검증 및 비용 최적화 방법을 익힌다

### ⏱️ 예상 소요 시간: 90분

### 💰 예상 API 비용: ~$0.5~1.0 (GPT-4o-mini 기준)

In [1]:
# 환경 설정
import json
import os
import time
import random
from typing import List, Dict
from dotenv import load_dotenv

load_dotenv()

print("✅ Session 15: 합성 데이터 생성 & Distillation")
print("📌 이 노트북은 OpenAI API를 사용합니다 (GPU 불필요)")
print()

# 데이터 디렉토리
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "samples")
os.makedirs(DATA_DIR, exist_ok=True)

# OpenAI API 설정
try:
    from openai import OpenAI
    client = OpenAI()  # OPENAI_API_KEY 환경변수 사용
    print("✅ OpenAI 클라이언트 초기화 성공")
except ImportError:
    print("⚠️ openai 라이브러리가 설치되어 있지 않습니다.")
    print("   설치: pip install openai")
    client = None
except Exception as e:
    print(f"⚠️ OpenAI 클라이언트 초기화 실패: {e}")
    print("   OPENAI_API_KEY 환경변수를 설정하세요.")
    client = None

✅ Session 15: 합성 데이터 생성 & Distillation
📌 이 노트북은 OpenAI API를 사용합니다 (GPU 불필요)

✅ OpenAI 클라이언트 초기화 성공


---
## 🎯 1. 합성 데이터와 Distillation 개요

### 합성 데이터(Synthetic Data)란?

실제 데이터가 아닌, **AI 모델이 생성한 인공 데이터**를 의미합니다.

```
🧠 Teacher Model (GPT-4)
    ↓ 데이터 생성
📚 합성 학습 데이터 (수천~수만개)
    ↓ 학습
🤖 Student Model (Qwen2.5-1.5B)
```

### Knowledge Distillation이란?

큰 모델(Teacher)의 **지식을 작은 모델(Student)**에게 전달하는 기법

| 항목 | Teacher | Student |
|------|---------|----------|
| 모델 | GPT-4, Claude | Qwen2.5-1.5B |
| 파라미터 | 수천억 | 15억 |
| 비용 | API 비용 | 로컬 무료 |
| 속도 | 느림 | 빠름 |
| 프라이버시 | 외부 서버 | 로컬 환경 |

---
## 1️⃣ 2. 합성 데이터 생성이 필요한 이유

### 현실적인 문제들

| 문제 | 설명 | 합성 데이터 해결 |
|------|------|----------------|
| 📌 **데이터 부족** | 도메인 특화 데이터가 적음 | LLM으로 대량 생성 |
| 📌 **비용 문제** | 전문가 라벨링 비용 높음 | API 비용으로 대체 |
| 📌 **프라이버시** | 실제 데이터 활용 제한 | 합성 데이터로 우회 |
| 📌 **다양성 부족** | 편향된 데이터 | 다양한 시나리오 생성 |
| 📌 **시간 문제** | 데이터 수집에 수개월 | 수시간~수일 |

### 합성 데이터의 성공 사례

- 🏆 **Alpaca (Stanford)**: GPT-3.5로 52K 데이터 생성 → 7B 모델 파인튜닝
- 🏆 **Vicuna**: ShareGPT 대화 데이터 → 13B 모델이 GPT-4의 90% 성능
- 🏆 **Orca**: GPT-4의 추론 과정 학습 → 작은 모델의 추론 능력 향상
- 🏆 **phi-1.5 (Microsoft)**: 합성 교과서 데이터로 1.3B 모델이 5~25B 모델 성능

---
## 2️⃣ 3. Self-Instruct 방법론

### Self-Instruct 프로세스

2022년 Wang et al.이 제안한 방법으로, LLM이 스스로 학습 데이터를 생성하는 부트스트래핑 기법입니다.

```
📝 Seed Instructions (초기 시드 데이터 ~175개)
    ↓
🤖 LLM이 새로운 Instruction 생성
    ↓
🔍 품질 필터링 (중복 제거, 유사도 체크)
    ↓
🤖 LLM이 각 Instruction에 대한 Output 생성
    ↓
📚 새로운 학습 데이터 (52K+)
    ↓
🔄 반복 (새 데이터를 시드에 추가)
```

### 핵심 원리

1. **다양성 확보**: 시드 데이터의 카테고리를 다양하게
2. **품질 필터링**: 생성된 데이터 중 고품질만 선별
3. **반복 확장**: 좋은 데이터를 시드에 추가하여 재귀적 확장

In [2]:
# 📝 Seed 데이터 로드
print("📝 Seed 데이터 로드")
print("=" * 60)

# data/samples/alpaca_ko_sample.json 에서 Seed 로드
seed_file = os.path.join(DATA_DIR, "alpaca_ko_sample.json")

with open(seed_file, 'r', encoding='utf-8') as f:
    full_data = json.load(f)

print(f"📂 로드된 데이터: {seed_file}")
print(f"   총 {len(full_data)}건")

# Seed로 사용할 5개만 선택
seed_data = full_data[:5]

print(f"\n🌱 Seed 데이터 ({len(seed_data)}개):")
for i, item in enumerate(seed_data, 1):
    print(f"  {i}. {item['instruction'][:60]}")
    if item.get('input'):
        print(f"     입력: {item['input'][:40]}")
    print(f"     출력: {item['output'][:60]}...")

📝 Seed 데이터 로드
📂 로드된 데이터: /home/yskim/LLM_Advanced/data/samples/alpaca_ko_sample.json
   총 10건

🌱 Seed 데이터 (5개):
  1. 다음 문장을 영어로 번역하세요.
     입력: 오늘 날씨가 매우 좋습니다.
     출력: The weather is very nice today....
  2. 다음 텍스트의 감성을 분석하세요.
     입력: 이 영화는 정말 재미있었어요! 배우들의 연기도 훌륭했습니다.
     출력: 긍정적인 감성입니다. '재미있었어요', '훌륭했습니다' 등의 긍정적 표현이 사용되었습니다....
  3. 주어진 키워드로 짧은 이야기를 작성하세요.
     입력: 고양이, 비, 카페
     출력: 비가 오는 오후, 작은 카페 창가에 고양이 한 마리가 앉아 있었습니다. 빗소리를 들으며 따뜻한 카페 안에서 ...
  4. 파이썬에서 리스트 컴프리헨션의 사용법을 설명하세요.
     출력: 리스트 컴프리헨션은 파이썬에서 리스트를 간결하게 생성하는 방법입니다.

기본 문법: [expression f...
  5. 다음 문장을 3줄로 요약하세요.
     입력: 인공지능(AI)은 1950년대 앨런 튜링의 연구에서 시작되어 오랜 역사를
     출력: 1. AI는 1950년대부터 시작된 오랜 역사를 가진 분야입니다.
2. 2017년 트랜스포머의 등장으로 자연...


---
## 3️⃣ 4. GPT-4를 활용한 데이터 생성 (OpenAI API)

In [3]:
# 🔧 데이터 생성 프롬프트 + API 호출 함수
print("🔧 데이터 생성 함수 정의")
print("=" * 60)

SYSTEM_PROMPT = """당신은 AI 학습 데이터 생성 전문가입니다.
주어진 시드 데이터를 참고하여, 새로운 instruction-output 쌍을 생성합니다.

규칙:
1. 시드 데이터와 유사하지만 다른 새로운 질문을 만드세요
2. 한국어로 작성하세요
3. output은 구체적이고 정확해야 합니다
4. 반드시 JSON 배열 형식으로만 출력하세요 (다른 텍스트 없이)"""


def generate_data_batch(client, seed_examples, n_generate=5, category=None):
    """GPT-4o-mini로 학습 데이터 배치 생성"""
    # Seed 예시 → 프롬프트 구성
    examples = random.sample(seed_examples, min(3, len(seed_examples)))
    example_text = json.dumps([
        {"instruction": e["instruction"], "input": e.get("input", ""), "output": e["output"]}
        for e in examples
    ], ensure_ascii=False, indent=2)
    
    category_hint = f"\n카테고리: {category}" if category else ""
    
    prompt = f"""다음은 기존 학습 데이터 예시입니다:

{example_text}

위 예시를 참고하여, 새로운 {n_generate}개의 instruction-output 쌍을 생성하세요.{category_hint}
반드시 JSON 배열 형식으로만 출력하세요.
[{{"instruction": "...", "input": "...", "output": "..."}}, ...]"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.8,
        max_tokens=2000,
    )
    
    content = response.choices[0].message.content.strip()
    # 코드 블록 제거
    if content.startswith("```"):
        content = content.split("```")[1]
        if content.startswith("json"):
            content = content[4:]
    
    generated = json.loads(content)
    usage = response.usage
    
    return generated, {
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
    }

print("✅ generate_data_batch() 정의 완료")

🔧 데이터 생성 함수 정의
✅ generate_data_batch() 정의 완료


In [4]:
# 🚀 테스트: 1개 배치 생성해보기
print("🚀 테스트: Seed 5개 → 새 데이터 5개 생성")
print("=" * 60)

generated, usage = generate_data_batch(client, seed_data, n_generate=5)

print(f"✅ {len(generated)}개 생성 완료!")
print(f"   토큰 사용: {usage['total_tokens']:,}")
print()

for i, item in enumerate(generated, 1):
    print(f"  {i}. {item['instruction'][:70]}")
    if item.get('input'):
        print(f"     입력: {item['input'][:50]}")
    print(f"     출력: {item['output'][:80]}...")
    print()

🚀 테스트: Seed 5개 → 새 데이터 5개 생성
✅ 5개 생성 완료!
   토큰 사용: 1,061

  1. 다음 문장의 주제를 파악하세요.
     입력: 많은 사람들이 여행을 통해 새로운 문화를 경험하고, 다양한 사람들을 만나며 삶의 질을 높이
     출력: 주제는 여행의 중요성과 그로 인한 문화 경험입니다....

  2. 주어진 단어로 시를 작성하세요.
     입력: 가을, 바람, 나무
     출력: 가을 바람이 살랑살랑,
나무의 잎들이 춤을 춥니다.
황금빛으로 물든 풍경 속에서,
자연의 노래가 들려옵니다....

  3. 주어진 사건을 통해 배운 교훈을 서술하세요.
     입력: 친구와의 약속을 잊고 혼자 시간을 보낸 날
     출력: 이번 사건을 통해, 약속의 중요성과 친구와의 관계를 소중히 여기는 것이 얼마나 중요한지를 배웠습니다....

  4. 다음 텍스트의 주요 내용을 정리하세요.
     입력: 소셜 미디어의 발전은 사람들 간의 소통 방식을 변화시키고 있습니다. 이제는 전 세계 어디서
     출력: 1. 소셜 미디어는 소통 방식을 변화시켰습니다.
2. 전 세계 어디서든 즉시 정보 공유가 가능합니다.
3. 하지만 사생활 침해와 같은 부작용도 ...

  5. 다음 상황에 대한 해결책을 제안하세요.
     입력: 직장에서 팀원 간의 의사소통이 원활하지 않아 프로젝트 진행에 어려움을 겪고 있습니다.
     출력: 정기적인 팀 미팅을 통해 의견을 나누고, 각 팀원의 역할과 책임을 명확히 하여 의사소통을 개선할 수 있습니다....



---
## 4️⃣ 5. 실습: Seed 기반 대규모 확장 (5개 → 30개)

카테고리별로 데이터를 자동 생성하는 파이프라인을 실행합니다.

In [5]:
# 🚀 카테고리별 데이터 생성 파이프라인
print("🚀 카테고리별 데이터 생성 파이프라인")
print("=" * 60)

categories = ["번역", "코딩", "요약"]  # 3개 카테고리 × 10개 = 30개

all_generated = []
total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

print(f"📌 {len(categories)}개 카테고리 × 10개 = 총 {len(categories)*10}개 생성")
print(f"📌 모델: gpt-4o-mini")
print()

for i, category in enumerate(categories):
    print(f"  [{i+1}/{len(categories)}] {category}...", end=" ")
    
    generated, usage = generate_data_batch(
        client, seed_data, n_generate=10, category=category
    )
    
    for item in generated:
        item["category"] = category
    all_generated.extend(generated)
    
    for key in total_usage:
        total_usage[key] += usage.get(key, 0)
    
    print(f"✅ {len(generated)}개")
    time.sleep(1)

# 비용 계산
input_cost = total_usage['prompt_tokens'] / 1_000_000 * 0.15
output_cost = total_usage['completion_tokens'] / 1_000_000 * 0.60

print(f"\n{'='*60}")
print(f"📊 결과: 총 {len(all_generated)}개 생성")
print(f"   토큰: {total_usage['total_tokens']:,}")
print(f"   비용: ${input_cost + output_cost:.4f}")

🚀 카테고리별 데이터 생성 파이프라인
📌 3개 카테고리 × 10개 = 총 30개 생성
📌 모델: gpt-4o-mini

  [1/3] 번역... ✅ 10개
  [2/3] 코딩... ✅ 10개
  [3/3] 요약... ✅ 10개

📊 결과: 총 30개 생성
   토큰: 3,468
   비용: $0.0014


In [6]:
# 생성 데이터 확인
print("📋 생성된 데이터 샘플 확인")
print("=" * 60)

if all_generated:
    # 카테고리별 분포
    from collections import Counter
    cat_counter = Counter(item.get('category', '미분류') for item in all_generated)
    
    print("📊 카테고리별 분포:")
    for cat, count in sorted(cat_counter.items(), key=lambda x: -x[1]):
        bar = "█" * count
        print(f"  {cat:>10s}: {count:>3d} {bar}")
    
    # 랜덤 샘플 표시
    print(f"\n📋 랜덤 샘플 3개:")
    samples = random.sample(all_generated, min(3, len(all_generated)))
    for i, sample in enumerate(samples, 1):
        print(f"\n  {'─'*50}")
        print(f"  📌 샘플 {i} [{sample.get('category', '미분류')}]")
        print(f"  Instruction: {sample['instruction'][:80]}")
        if sample.get('input', '').strip():
            print(f"  Input: {sample['input'][:80]}")
        print(f"  Output: {sample['output'][:100]}...")
else:
    print("⚠️ 생성된 데이터가 없습니다.")

📋 생성된 데이터 샘플 확인
📊 카테고리별 분포:
          번역:  10 ██████████
          코딩:  10 ██████████
          요약:  10 ██████████

📋 랜덤 샘플 3개:

  ──────────────────────────────────────────────────
  📌 샘플 1 [코딩]
  Instruction: 주어진 JSON 데이터를 파싱하는 코드를 작성하세요.
  Input: {"name": "홍길동", "age": 30}
  Output: import json

data = json.loads('{"name": "홍길동", "age": 30}')...

  ──────────────────────────────────────────────────
  📌 샘플 2 [요약]
  Instruction: 주어진 문장의 요지를 간단히 정리하세요.
  Input: 지구 온난화는 인류의 활동으로 인해 발생하고 있으며, 이에 대한 대응이 필요하다.
  Output: 지구 온난화는 인간 활동에 의해 발생하며, 대응이 필요하다....

  ──────────────────────────────────────────────────
  📌 샘플 3 [번역]
  Instruction: 다음 문장을 영어로 번역하세요.
  Input: 이 프로젝트는 다음 달에 마무리될 것입니다.
  Output: This project will be completed next month....


---
## 5️⃣ 6. Distillation: 큰 모델 → 작은 모델 지식 전달

### Distillation의 두 가지 방식

#### 방식 1: 데이터 Distillation (우리가 할 것)
```
GPT-4 (Teacher) → 고품질 응답 생성 → 학습 데이터
                                         ↓
Qwen-1.5B (Student) ← SFT 학습 ← 학습 데이터
```

#### 방식 2: Logit Distillation (전통적 방식)
```
Teacher Model → Soft Labels (확률 분포)
                     ↓
Student Model → 확률 분포를 학습 (KL Divergence)
```

### 데이터 Distillation 전략

| 전략 | 설명 | 효과 |
|------|------|------|
| **Simple Distillation** | GPT-4의 응답을 그대로 학습 | 기본 |
| **Chain-of-Thought** | 추론 과정까지 포함하여 생성 | 추론 능력 향상 |
| **Step-by-Step** | 단계별 설명을 포함 | 설명 능력 향상 |
| **Critique + Revision** | 자가 비평 + 수정 과정 포함 | 자기교정 능력 |

In [7]:
# 🔧 Distillation 3가지 전략 비교 (실제 API 호출)
print("🔧 Distillation: 동일 질문 → 3가지 전략으로 응답 생성")
print("=" * 60)

question = "트랜스포머 모델의 어텐션 메커니즘을 설명해주세요."
print(f"❓ 질문: {question}\n")

strategies = {
    "Simple": "당신은 유능한 AI 어시스턴트입니다. 정확하고 유용한 답변을 제공하세요.",
    "CoT": """당신은 전문 교육자입니다.
질문에 답할 때 먼저 생각 과정을 [생각 과정] 섹션에 보여주고, 그 다음 [답변] 섹션에 최종 답변을 제공하세요.""",
    "Step-by-Step": """당신은 전문 교육자입니다.
질문에 답할 때 단계별로 상세하게 설명하세요. 각 단계는 '1단계:', '2단계:' 형식으로 구분하세요.""",
}

distill_comparison = {}

for name, system_prompt in strategies.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        temperature=0.7,
        max_tokens=500,
    )
    answer = response.choices[0].message.content.strip()
    distill_comparison[name] = answer
    
    print(f"📋 전략: {name}")
    print(f"{'─'*50}")
    print(f"{answer[:300]}...")
    print()

print("💡 CoT/Step-by-Step 전략은 작은 모델의 추론 능력을 향상시킵니다.")
print("   Orca 논문에서 이 접근법의 효과가 입증되었습니다.")

🔧 Distillation: 동일 질문 → 3가지 전략으로 응답 생성
❓ 질문: 트랜스포머 모델의 어텐션 메커니즘을 설명해주세요.

📋 전략: Simple
──────────────────────────────────────────────────
트랜스포머 모델의 어텐션 메커니즘은 자연어 처리(NLP)에서 매우 중요한 역할을 합니다. 이 메커니즘은 입력 시퀀스의 모든 단어가 서로에게 주의를 기울일 수 있도록 하여, 문맥 정보를 효과적으로 캡처합니다. 어텐션 메커니즘의 핵심 요소는 다음과 같습니다.

### 1. 어텐션의 기본 개념
어텐션 메커니즘은 입력 시퀀스의 각 단어(또는 토큰)가 다른 단어에 얼마나 주의를 기울여야 하는지를 계산합니다. 이를 통해 모델은 중요한 정보에 집중하고 덜 중요한 정보는 무시할 수 있습니다.

### 2. 스케일드 도트 프로덕트 어텐션
트랜스포머...

📋 전략: CoT
──────────────────────────────────────────────────
[생각 과정]
트랜스포머 모델의 어텐션 메커니즘은 자연어 처리 및 다양한 시퀀스 데이터 처리에서 중요한 역할을 합니다. 어텐션 메커니즘은 입력 시퀀스의 각 요소가 출력에 얼마나 영향을 미치는지를 결정하는 방법입니다. 특히, 트랜스포머에서는 '셀프 어텐션'이라는 개념을 사용하여 입력 시퀀스 내에서 각 단어가 서로 어떤 관계를 가지는지를 평가합니다. 이를 위해 세 가지 벡터, 즉 '쿼리(Query)', '키(Key)', '값(Value)'를 사용합니다. 

1. 입력 단어를 쿼리, 키, 값으로 변환합니다.
2. 각 쿼리와 모든 키 간의...

📋 전략: Step-by-Step
──────────────────────────────────────────────────
트랜스포머 모델의 어텐션 메커니즘은 자연어 처리(NLP) 분야에서 혁신적인 기술로, 입력 데이터 간의 관계를 효과적으로 파악하는 데 도움을 줍니다. 어텐션 메커니즘을 이해하기 위해 다음과 같은 단계로 나누어 설명하겠습

In [8]:
# 🚀 Distillation 데이터 대량 생성
print("🚀 CoT 전략으로 Distillation 데이터 생성")
print("=" * 60)

distill_questions = [
    "파이썬에서 데코레이터는 어떻게 동작하나요?",
    "GPT와 BERT의 차이점은 무엇인가요?",
    "LoRA 파인튜닝이 Full Fine-Tuning보다 효율적인 이유는?",
    "RAG 시스템에서 벡터 데이터베이스의 역할은?",
    "양자화(Quantization)가 모델 성능에 미치는 영향은?",
]

cot_system = """당신은 전문 교육자입니다.
질문에 답할 때 먼저 생각 과정을 보여주고, 그 다음 최종 답변을 제공하세요.
형식:
[생각 과정]
...
[답변]
..."""

distill_data = []
total_tokens = 0

for i, q in enumerate(distill_questions):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": cot_system},
            {"role": "user", "content": q}
        ],
        temperature=0.7,
        max_tokens=800,
    )
    answer = response.choices[0].message.content.strip()
    total_tokens += response.usage.total_tokens
    
    distill_data.append({
        "instruction": q,
        "input": "",
        "output": answer,
    })
    print(f"  [{i+1}/{len(distill_questions)}] ✅ {q[:40]}... ({len(answer)}자)")
    time.sleep(0.5)

print(f"\n📊 총 {len(distill_data)}개 생성, 토큰: {total_tokens:,}")
print(f"\n📋 샘플 (첫 번째):")
print(f"  Q: {distill_data[0]['instruction']}")
print(f"  A: {distill_data[0]['output'][:200]}...")

🚀 CoT 전략으로 Distillation 데이터 생성
  [1/5] ✅ 파이썬에서 데코레이터는 어떻게 동작하나요?... (990자)
  [2/5] ✅ GPT와 BERT의 차이점은 무엇인가요?... (841자)
  [3/5] ✅ LoRA 파인튜닝이 Full Fine-Tuning보다 효율적인 이유는?... (912자)
  [4/5] ✅ RAG 시스템에서 벡터 데이터베이스의 역할은?... (914자)
  [5/5] ✅ 양자화(Quantization)가 모델 성능에 미치는 영향은?... (907자)

📊 총 5개 생성, 토큰: 2,629

📋 샘플 (첫 번째):
  Q: 파이썬에서 데코레이터는 어떻게 동작하나요?
  A: [생각 과정]
데코레이터는 파이썬에서 함수나 메서드의 동작을 수정하거나 확장하는 방법입니다. 데코레이터는 보통 다른 함수를 인자로 받아 새로운 함수를 반환하는 형태로 구현됩니다. 이를 통해 기존 함수에 추가적인 기능을 부여할 수 있습니다. 데코레이터는 주로 로깅, 접근 제어, 성능 측정 등 다양한 목적으로 사용됩니다.

데코레이터의 기본 구조는 함수 정의에...


---
## 6️⃣ 7. 생성 데이터 품질 검증 및 필터링

In [9]:
# 🔍 품질 검증 + 최종 데이터 저장
print("🔍 생성 데이터 품질 검증 및 저장")
print("=" * 60)

def quality_filter(data, min_output_len=20):
    """생성 데이터 품질 필터링"""
    filtered = []
    seen = set()
    removed = {"empty": 0, "short": 0, "duplicate": 0}
    
    for item in data:
        instruction = item.get("instruction", "").strip()
        output = item.get("output", "").strip()
        
        if not instruction or not output:
            removed["empty"] += 1
            continue
        if len(output) < min_output_len:
            removed["short"] += 1
            continue
        if instruction in seen:
            removed["duplicate"] += 1
            continue
        seen.add(instruction)
        filtered.append(item)
    
    return filtered, removed

# 합성 데이터 필터링
print("1️⃣ 합성 데이터 (Self-Instruct)")
filtered_synthetic, stats1 = quality_filter(all_generated)
print(f"   입력: {len(all_generated)}개 → 통과: {len(filtered_synthetic)}개")
print(f"   제거: 빈 데이터 {stats1['empty']}, 짧은 응답 {stats1['short']}, 중복 {stats1['duplicate']}")

# Distillation 데이터 필터링
print("\n2️⃣ Distillation 데이터 (CoT)")
filtered_distill, stats2 = quality_filter(distill_data)
print(f"   입력: {len(distill_data)}개 → 통과: {len(filtered_distill)}개")

# 통합 저장
final_data = seed_data + filtered_synthetic + filtered_distill

# instruction, input, output만 저장
save_data = [
    {k: v for k, v in item.items() if k in ["instruction", "input", "output"]}
    for item in final_data
]

output_file = os.path.join(DATA_DIR, "synthetic_data.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print(f"✅ 최종 저장: {output_file}")
print(f"   Seed: {len(seed_data)}개 + 합성: {len(filtered_synthetic)}개 + Distill: {len(filtered_distill)}개")
print(f"   합계: {len(save_data)}개")

🔍 생성 데이터 품질 검증 및 저장
1️⃣ 합성 데이터 (Self-Instruct)
   입력: 30개 → 통과: 21개
   제거: 빈 데이터 0, 짧은 응답 0, 중복 9

2️⃣ Distillation 데이터 (CoT)
   입력: 5개 → 통과: 5개

✅ 최종 저장: /home/yskim/LLM_Advanced/data/samples/synthetic_data.json
   Seed: 5개 + 합성: 21개 + Distill: 5개
   합계: 31개


---
## 7️⃣ 8. 비용 계산 및 최적화 팁

In [10]:
# API 비용 계산기
print("💰 데이터 생성 비용 계산기")
print("=" * 60)

# 모델별 가격 (2024년 기준, 100만 토큰당)
pricing = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4-turbo": {"input": 10.00, "output": 30.00},
    "gpt-3.5-turbo": {"input": 0.50, "output": 1.50},
}

def estimate_cost(n_samples, avg_input_tokens=500, avg_output_tokens=300, model="gpt-4o-mini"):
    """데이터 생성 비용 추정"""
    prices = pricing.get(model, pricing["gpt-4o-mini"])
    
    total_input = n_samples * avg_input_tokens
    total_output = n_samples * avg_output_tokens
    
    input_cost = total_input / 1_000_000 * prices["input"]
    output_cost = total_output / 1_000_000 * prices["output"]
    total_cost = input_cost + output_cost
    
    return {
        "model": model,
        "n_samples": n_samples,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }

# 다양한 시나리오 비용 계산
scenarios = [
    (100, "gpt-4o-mini", "소규모 실습"),
    (1000, "gpt-4o-mini", "중규모 프로젝트"),
    (10000, "gpt-4o-mini", "대규모 프로젝트"),
    (1000, "gpt-4o", "고품질 데이터 (gpt-4o)"),
    (1000, "gpt-4-turbo", "최고 품질 (gpt-4-turbo)"),
]

print(f"\n{'시나리오':>25s} | {'데이터 수':>10s} | {'모델':>15s} | {'예상 비용':>12s}")
print("-" * 75)

for n, model, desc in scenarios:
    result = estimate_cost(n, model=model)
    print(f"{desc:>25s} | {n:>10,d} | {model:>15s} | ${result['total_cost']:>10.2f}")

print("\n💡 비용 최적화 팁:")
print("  1️⃣ gpt-4o-mini 사용 → gpt-4o 대비 약 10~20배 저렴")
print("  2️⃣ 배치 생성: 한 번에 여러 개 생성하면 프롬프트 토큰 절약")
print("  3️⃣ 프롬프트 최적화: 시드 예시 수를 최소화")
print("  4️⃣ 점진적 접근: 100개 → 검증 → 1000개 → 검증 → 확장")
print("  5️⃣ 캐싱: 동일 프롬프트 재사용 방지")

💰 데이터 생성 비용 계산기

                     시나리오 |      데이터 수 |              모델 |        예상 비용
---------------------------------------------------------------------------
                   소규모 실습 |        100 |     gpt-4o-mini | $      0.03
                 중규모 프로젝트 |      1,000 |     gpt-4o-mini | $      0.26
                 대규모 프로젝트 |     10,000 |     gpt-4o-mini | $      2.55
         고품질 데이터 (gpt-4o) |      1,000 |          gpt-4o | $      4.25
      최고 품질 (gpt-4-turbo) |      1,000 |     gpt-4-turbo | $     14.00

💡 비용 최적화 팁:
  1️⃣ gpt-4o-mini 사용 → gpt-4o 대비 약 10~20배 저렴
  2️⃣ 배치 생성: 한 번에 여러 개 생성하면 프롬프트 토큰 절약
  3️⃣ 프롬프트 최적화: 시드 예시 수를 최소화
  4️⃣ 점진적 접근: 100개 → 검증 → 1000개 → 검증 → 확장
  5️⃣ 캐싱: 동일 프롬프트 재사용 방지


In [11]:
# 비용 대비 효과 분석
print("📊 데이터 양 vs 성능 (일반적 트렌드)")
print("=" * 60)

print("""
📌 일반적으로 관찰되는 데이터 양 vs 파인튜닝 성능:

  데이터 수  | 예상 품질  | 비용 (4o-mini) | 권장 단계
  ─────────┼──────────┼──────────────┼──────────
    100개   | ★★☆☆☆   |    ~$0.05    | 1. 프로토타이핑
    500개   | ★★★☆☆   |    ~$0.25    | 2. 초기 실험
  1,000개   | ★★★★☆   |    ~$0.50    | 3. 기본 파인튜닝
  5,000개   | ★★★★☆   |    ~$2.50    | 4. 준 프로덕션
 10,000개   | ★★★★★   |    ~$5.00    | 5. 프로덕션
 50,000개   | ★★★★★   |   ~$25.00    | 6. 고성능 (수확체감)

💡 핵심 인사이트:
  - 1,000~5,000개에서 가성비가 가장 좋음
  - 10,000개 이상부터는 수확체감(diminishing returns)
  - 데이터 품질이 양보다 중요 (LIMA 논문)
  - 도메인이 좁을수록 적은 데이터로도 효과적
""")

📊 데이터 양 vs 성능 (일반적 트렌드)

📌 일반적으로 관찰되는 데이터 양 vs 파인튜닝 성능:

  데이터 수  | 예상 품질  | 비용 (4o-mini) | 권장 단계
  ─────────┼──────────┼──────────────┼──────────
    100개   | ★★☆☆☆   |    ~$0.05    | 1. 프로토타이핑
    500개   | ★★★☆☆   |    ~$0.25    | 2. 초기 실험
  1,000개   | ★★★★☆   |    ~$0.50    | 3. 기본 파인튜닝
  5,000개   | ★★★★☆   |    ~$2.50    | 4. 준 프로덕션
 10,000개   | ★★★★★   |    ~$5.00    | 5. 프로덕션
 50,000개   | ★★★★★   |   ~$25.00    | 6. 고성능 (수확체감)

💡 핵심 인사이트:
  - 1,000~5,000개에서 가성비가 가장 좋음
  - 10,000개 이상부터는 수확체감(diminishing returns)
  - 데이터 품질이 양보다 중요 (LIMA 논문)
  - 도메인이 좁을수록 적은 데이터로도 효과적



---
## 📝 9. 정리 및 핵심 요약

### 🎯 이번 세션에서 배운 핵심 개념

| # | 개념 | 핵심 내용 |
|---|------|----------|
| 1️⃣ | **합성 데이터** | LLM이 생성한 인공 학습 데이터 |
| 2️⃣ | **Self-Instruct** | Seed → LLM 생성 → 필터링 → 반복 |
| 3️⃣ | **GPT-4 생성** | 프롬프트 설계 + API 호출 + JSON 파싱 |
| 4️⃣ | **배치 확장** | 10개 시드 → 100개+ 자동 생성 |
| 5️⃣ | **Distillation** | Teacher(GPT-4) → Student(Qwen-1.5B) 지식 전달 |
| 6️⃣ | **품질 필터링** | 길이/중복/품질 자동 검증 |
| 7️⃣ | **비용 최적화** | gpt-4o-mini 사용, 배치 생성, 점진적 확장 |

### 🔑 실무 체크리스트

```
□ 시드 데이터 준비 (카테고리별 최소 3~5개)
□ 생성 프롬프트 설계 및 테스트 (100개 먼저 생성)
□ 품질 검증 (수동 검토 + 자동 필터링)
□ 대규모 생성 (1,000~5,000개)
□ 최종 품질 필터링 및 데이터 포맷 변환
□ 비용 추적 및 최적화
```

### 📚 다음 세션 예고
- **Session 16**: Next Token Prediction 기반 SFT 실습 - 실제 모델 파인튜닝!

In [12]:
print("\n✅ Session 15 완료!")
print("📚 다음 세션: Next Token Prediction 기반 SFT 실습")
print(f"\n📁 생성된 파일:")
print(f"  1. {os.path.join(DATA_DIR, 'synthetic_data.json')} (합성 데이터)")
print("\n💡 다음 세션에서는 이 데이터를 사용하여 실제 모델을 파인튜닝합니다!")
print("   🔧 준비물: GPU (RTX 4060), Qwen2.5-1.5B-Instruct 모델")


✅ Session 15 완료!
📚 다음 세션: Next Token Prediction 기반 SFT 실습

📁 생성된 파일:
  1. /home/yskim/LLM_Advanced/data/samples/synthetic_data.json (합성 데이터)

💡 다음 세션에서는 이 데이터를 사용하여 실제 모델을 파인튜닝합니다!
   🔧 준비물: GPU (RTX 4060), Qwen2.5-1.5B-Instruct 모델
